[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week1_data_foundations/day06_outliers/day06_notebook.ipynb)

# Day 6 / 42: Outliers
### 42 Days of ML Challenge | @VaishnaviJagtap18

---

## What You'll Learn
- The 3 standard outlier detection methods: IQR, Z-score, Isolation Forest
- When each method works and where it breaks
- The most important decision in outlier handling: remove vs keep vs flag
- How outlier strategy changes model performance
- A real fraud detection exercise using Isolation Forest

---

## The Core Concept

An outlier is a data point that sits far from the rest of your data. That's the statistical definition. But whether you remove it, keep it, or treat it as a signal depends entirely on your domain.

**In house price prediction**, a ₹50 crore mansion in a dataset of ₹50 lakh homes is noise. It distorts your model and produces worse predictions for the 99% of typical homes.

**In fraud detection at Visa**, a ₹50 crore transaction is the exact thing your model needs to catch. Remove it and you've trained a model that misses fraud.

Same statistical outlier. Opposite action. Domain context decides, not the algorithm.

**Visa's fraud team** published research showing that Isolation Forest combined with supervised models catches fraud that pure supervised methods miss, because fraud is by definition statistically anomalous behaviour. Their production system processes 65,000 transactions per second and flags outliers in under 1 millisecond.

---

## Setup

In [ ]:
# Install dependencies (only needed in Colab)
# !pip install pyod -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("All imports successful. Ready for Day 6.")

---

## Step 1: Create the Dataset

We'll simulate a retail sales dataset with realistic data plus injected outliers, so you know exactly which rows are outliers and can verify that each method finds them.

In [ ]:
np.random.seed(42)

# Normal sales data: ~300 rows, normally distributed around 5000
n_normal = 300
normal_sales = np.random.normal(loc=5000, scale=800, size=n_normal)

# Injected outliers: extreme high (data entry errors or genuinely rare bulk orders)
# and extreme low (possible returns or data corruption)
high_outliers = np.array([18000, 19500, 21000, 20000])   # extreme high
low_outliers  = np.array([500, 200, 300, 100])            # extreme low

all_sales = np.concatenate([normal_sales, high_outliers, low_outliers])
np.random.shuffle(all_sales)

df = pd.DataFrame({
    'sales':       all_sales,
    'units_sold':  all_sales / 50 + np.random.normal(0, 5, len(all_sales)),
    'store_id':    np.random.randint(1, 20, len(all_sales))
})

print(f"Dataset shape: {df.shape}")
print(f"Injected outliers: {len(high_outliers) + len(low_outliers)}")
print("\nDescriptive stats:")
print(df['sales'].describe().round(2))

---

## Step 2: Visualise the Data Before Detection

Always plot your data before running any detection algorithm. The boxplot and histogram together tell you whether outliers are present and roughly where they sit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['sales'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Sales Distribution (Raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sales (₹)')
axes[0].set_ylabel('Count')

# Boxplot
axes[1].boxplot(df['sales'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Boxplot: Outliers Are the Dots Beyond Whiskers', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Sales (₹)')
axes[1].set_xticks([])

plt.tight_layout()
plt.suptitle('Step 2: Raw Data Before Any Outlier Detection', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print("Notice the dots beyond the whiskers in the boxplot — those are your outliers.")
print("The histogram shows the bulk of data near 5000 and a thin tail extending to 20000+.")

---

## Step 3: Method 1 — IQR (Interquartile Range)

**How it works:** Any value below `Q1 - 1.5 * IQR` or above `Q3 + 1.5 * IQR` is flagged.

**Strength:** Non-parametric. Makes no assumption about the distribution. Works on skewed data.

**Weakness:** Only looks at one feature at a time. A transaction of ₹20,000 at 3am is suspicious. A transaction of ₹20,000 during a Diwali sale weekend might not be. IQR cannot use that context.

In [ ]:
Q1  = df['sales'].quantile(0.25)
Q3  = df['sales'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

iqr_outlier_mask = (df['sales'] < lower_bound) | (df['sales'] > upper_bound)

print(f"Q1:          {Q1:.0f}")
print(f"Q3:          {Q3:.0f}")
print(f"IQR:         {IQR:.0f}")
print(f"Lower bound: {lower_bound:.0f}")
print(f"Upper bound: {upper_bound:.0f}")
print(f"\nOutliers flagged by IQR: {iqr_outlier_mask.sum()}")
print("\nOutlier values:")
print(df.loc[iqr_outlier_mask, 'sales'].sort_values().values.round(0))

---

## Step 4: Method 2 — Z-Score

**How it works:** Converts each value to how many standard deviations it sits from the mean. Anything beyond ±3 is flagged.

**Strength:** Simple, fast, widely understood.

**Weakness:** Assumes the data is roughly normally distributed. On heavily skewed data, the mean and std are themselves pulled by outliers, so the method becomes unreliable. Also misses multivariate outliers — a data point can have a normal sales value AND a normal hour value individually, but be anomalous when both are considered together.

In [ ]:
z_scores         = np.abs(stats.zscore(df['sales']))
z_outlier_mask   = z_scores > 3

print(f"Z-score threshold: ±3 standard deviations")
print(f"Mean sales:  {df['sales'].mean():.0f}")
print(f"Std sales:   {df['sales'].std():.0f}")
print(f"\nOutliers flagged by Z-score: {z_outlier_mask.sum()}")
print("\nOutlier values:")
print(df.loc[z_outlier_mask, 'sales'].sort_values().values.round(0))

print("\n--- Why Z-score flags fewer than IQR ---")
print("The mean and std are pulled upward by the high outliers themselves.")
print("This raises the ±3 sigma threshold, making some outliers look less extreme.")
print("This is called 'masking' — extreme values hide each other in Z-score.")

---

## Step 5: Method 3 — Isolation Forest

**How it works:** Builds random decision trees. Points that get isolated in fewer splits are anomalies. It works across multiple features simultaneously — this is what the other two methods cannot do.

**Strength:** Handles multivariate outliers, makes no distributional assumption, scales well.

**Weakness:** `contamination` is a hyperparameter you set manually. If you say 5%, it flags 5% of your data regardless. You need domain knowledge to set this reasonably. It's also less interpretable than IQR or Z-score.

In [ ]:
# contamination = estimated proportion of outliers in your data
# We injected 8 outliers out of 308 total => ~2.6%, so 0.025 is reasonable
iso_forest = IsolationForest(contamination=0.025, random_state=42)

# Use multiple features — this is where Isolation Forest shines
iso_pred           = iso_forest.fit_predict(df[['sales', 'units_sold']])
iso_outlier_mask   = iso_pred == -1   # -1 = outlier, 1 = inlier

print(f"Outliers flagged by Isolation Forest: {iso_outlier_mask.sum()}")
print("\nOutlier values (sales):")
print(df.loc[iso_outlier_mask, 'sales'].sort_values().values.round(0))

# Anomaly scores: lower = more anomalous
df['anomaly_score'] = iso_forest.decision_function(df[['sales', 'units_sold']])
print(f"\nMost anomalous row (lowest score):\n{df.loc[df['anomaly_score'].idxmin()]}")

---

## Step 6: Compare All Three Methods Side by Side

In [ ]:
df['iqr_outlier'] = iqr_outlier_mask
df['z_outlier']   = z_outlier_mask
df['iso_outlier'] = iso_outlier_mask

comparison = pd.DataFrame({
    'Method':               ['IQR', 'Z-Score', 'Isolation Forest'],
    'Outliers Flagged':     [iqr_outlier_mask.sum(), z_outlier_mask.sum(), iso_outlier_mask.sum()],
    'Assumes Normal Dist':  ['No', 'Yes', 'No'],
    'Multivariate':         ['No', 'No', 'Yes'],
    'Hyperparameter':       ['1.5 * IQR (tunable)', 'Threshold ±3 (tunable)', 'contamination (must set)']
})

print(comparison.to_string(index=False))

print("\n--- Agreement between methods ---")
flagged_by_all = df['iqr_outlier'] & df['z_outlier'] & df['iso_outlier']
print(f"Rows flagged by ALL 3 methods: {flagged_by_all.sum()}")
print("If all 3 agree on a point, it's almost certainly a genuine outlier.")

---

## Step 7: Before vs After Boxplots

Visualise what your data looks like after removing IQR-flagged outliers.

In [ ]:
df_clean = df[~df['iqr_outlier']].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before
axes[0].boxplot(df['sales'], patch_artist=True,
                boxprops=dict(facecolor='#e74c3c', alpha=0.6),
                medianprops=dict(color='black', linewidth=2))
axes[0].set_title(f'BEFORE: {len(df)} rows\nContains {iqr_outlier_mask.sum()} IQR outliers',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sales (₹)')
axes[0].set_xticks([])

# After
axes[1].boxplot(df_clean['sales'], patch_artist=True,
                boxprops=dict(facecolor='#2ecc71', alpha=0.6),
                medianprops=dict(color='black', linewidth=2))
axes[1].set_title(f'AFTER IQR Removal: {len(df_clean)} rows\nOutliers removed',
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Sales (₹)')
axes[1].set_xticks([])

plt.suptitle('Step 7: Before vs After Outlier Removal (IQR Method)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Rows before: {len(df)}")
print(f"Rows after:  {len(df_clean)}")
print(f"Removed:     {len(df) - len(df_clean)} rows ({(len(df)-len(df_clean))/len(df)*100:.1f}%)")

---

## Step 8: Model Performance — Does Removing Outliers Actually Help?

This is the test that matters. Compare a Linear Regression model trained on raw data vs clean data.

In [ ]:
# Model on raw data
X_raw   = df[['units_sold']]
y_raw   = df['sales']
reg_raw = LinearRegression().fit(X_raw, y_raw)
rmse_raw = mean_squared_error(y_raw, reg_raw.predict(X_raw)) ** 0.5

# Model on cleaned data
X_clean   = df_clean[['units_sold']]
y_clean   = df_clean['sales']
reg_clean = LinearRegression().fit(X_clean, y_clean)
rmse_clean = mean_squared_error(y_clean, reg_clean.predict(X_clean)) ** 0.5

print("Linear Regression: units_sold -> sales")
print("-" * 45)
print(f"RMSE on raw data (with outliers):     {rmse_raw:.0f}")
print(f"RMSE on clean data (without outliers):{rmse_clean:.0f}")
print(f"Improvement:                          {((rmse_raw - rmse_clean)/rmse_raw*100):.1f}%")

print("\n--- What this tells you ---")
print("For prediction of typical sales, removing extreme values produces")
print("a model that fits the bulk of the data better.")
print("\nBut if your business needs accurate predictions FOR the extreme cases,")
print("you'd want a separate model or a robust regressor (HuberRegressor).")

---

## Step 9: Winsorizing — An Alternative to Removal

Removing outliers reduces your dataset size. Winsorizing caps extreme values at a threshold instead. The row stays. The extreme value gets replaced with the boundary value.

Use Winsorizing when:
- Losing rows is costly (small datasets)
- The data point is real, just extreme
- You're deploying a production pipeline where dropping rows can cause downstream issues

In [ ]:
from scipy.stats import mstats

# Cap bottom 1% and top 1%
df['sales_winsorized'] = mstats.winsorize(df['sales'], limits=[0.01, 0.01])

print("Original sales stats:")
print(df['sales'].describe().round(0))

print("\nWinsorized sales stats (1% cap on each end):")
print(df['sales_winsorized'].describe().round(0))

print(f"\nDataset size: {len(df)} rows (unchanged — no rows removed)")
print(f"Max before:   {df['sales'].max():.0f}")
print(f"Max after:    {df['sales_winsorized'].max():.0f}")

---

## Step 10: The Decision Framework

Before touching a single outlier, answer these questions.

In [ ]:
decision_framework = """
OUTLIER DECISION FRAMEWORK
==========================

Q1: Is the outlier a data error?
    (duplicate entry, sensor malfunction, wrong unit, fat-finger typo)
    -> YES: Correct or remove it. Not a real data point.
    -> NO:  Continue to Q2.

Q2: Is the outlier a genuine extreme event?
    (real bulk order, actual fraud transaction, legitimate high-value customer)
    -> YES: Continue to Q3.
    -> UNSURE: Check with domain experts before touching anything.

Q3: Does your model need to perform well on these extreme cases?
    -> YES (fraud detection, anomaly detection):
       KEEP outliers. They are your signal.
    -> NO (predict typical sales, house prices for average buyers):
       REMOVE or WINSORIZE. They add noise without adding value.

Q4: Is your dataset small?
    -> YES: Prefer Winsorizing over removal. Don't throw away data.
    -> NO:  Removal is fine if outliers are genuine noise.

Q5: Are you in production?
    -> YES: NEVER just drop rows. New data can bring new outliers.
       Build an outlier handling step into your pipeline:
       flag, cap, or route extreme values to a separate handler.
    -> NO (research/EDA): Removal is acceptable for exploration.
"""

print(decision_framework)

---

## Practice Exercise: Fraud Detection with Isolation Forest

You are given a transaction dataset. Your job: use Isolation Forest to flag suspicious transactions. Then evaluate how many real frauds your model caught.

**Instructions:** Run the cells. Read the output. Then answer the reflection questions at the end.

In [ ]:
# --- Generate the dataset ---
np.random.seed(42)

n_normal = 500
n_fraud  = 20

normal_txns = pd.DataFrame({
    'amount':        np.random.normal(200, 50, n_normal),   # typical transaction
    'hour':          np.random.randint(8, 22, n_normal),    # business hours
    'merchant_freq': np.random.randint(5, 30, n_normal)     # merchant used often
})
normal_txns['is_fraud'] = 0

fraud_txns = pd.DataFrame({
    'amount':        np.random.uniform(5000, 15000, n_fraud),  # unusually high
    'hour':          np.random.randint(0, 5, n_fraud),         # late night
    'merchant_freq': np.random.randint(1, 3, n_fraud)          # rare merchant
})
fraud_txns['is_fraud'] = 1

txn_df = pd.concat([normal_txns, fraud_txns], ignore_index=True).sample(frac=1, random_state=42)
txn_df = txn_df.reset_index(drop=True)

print(f"Total transactions: {len(txn_df)}")
print(f"Normal: {(txn_df['is_fraud']==0).sum()}")
print(f"Fraud:  {(txn_df['is_fraud']==1).sum()}")

In [ ]:
# --- Apply Isolation Forest ---
# contamination = approximate fraud rate = 20/520 ~ 0.038

features = ['amount', 'hour', 'merchant_freq']

iso_fraud = IsolationForest(contamination=0.04, random_state=42)
txn_df['prediction']    = iso_fraud.fit_predict(txn_df[features])
txn_df['flagged_fraud']  = (txn_df['prediction'] == -1).astype(int)

# Results
real_fraud_caught     = txn_df[(txn_df['is_fraud']==1) & (txn_df['flagged_fraud']==1)].shape[0]
false_alarms          = txn_df[(txn_df['is_fraud']==0) & (txn_df['flagged_fraud']==1)].shape[0]
total_flagged         = txn_df['flagged_fraud'].sum()
total_real_fraud      = txn_df['is_fraud'].sum()

print("=" * 45)
print("ISOLATION FOREST FRAUD DETECTION RESULTS")
print("=" * 45)
print(f"Total transactions:       {len(txn_df)}")
print(f"Total real fraud:         {total_real_fraud}")
print(f"Total flagged as fraud:   {total_flagged}")
print(f"Real fraud caught:        {real_fraud_caught}/{total_real_fraud} ({real_fraud_caught/total_real_fraud*100:.0f}%)")
print(f"False alarms:             {false_alarms}")
print(f"Precision of flags:       {real_fraud_caught/total_flagged*100:.0f}%")

In [ ]:
# --- Visualise flagged vs real fraud ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Amount vs Hour scatter
scatter_data = txn_df.copy()

# Normal (unflagged)
normal_pts  = scatter_data[scatter_data['flagged_fraud'] == 0]
flagged_pts = scatter_data[scatter_data['flagged_fraud'] == 1]

axes[0].scatter(normal_pts['hour'],   normal_pts['amount'],   c='steelblue', alpha=0.4, s=20, label='Normal')
axes[0].scatter(flagged_pts['hour'],  flagged_pts['amount'],  c='red',       alpha=0.8, s=60, label='Flagged', marker='X')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Transaction Amount (₹)')
axes[0].set_title('Flagged Transactions\n(Hour vs Amount)', fontweight='bold')
axes[0].legend()

# Confusion breakdown bar
categories = ['Real Fraud\nCaught', 'False\nAlarms', 'Real Fraud\nMissed']
values     = [real_fraud_caught, false_alarms, total_real_fraud - real_fraud_caught]
colors     = ['#2ecc71', '#e67e22', '#e74c3c']

bars = axes[1].bar(categories, values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(val), ha='center', fontweight='bold', fontsize=12)

axes[1].set_title('Isolation Forest Results Breakdown', fontweight='bold')
axes[1].set_ylabel('Count')

plt.suptitle('Fraud Detection Exercise: Isolation Forest Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Reflection Questions ---
# Answer these in your notebook or in the LinkedIn comments

questions = """
REFLECTION QUESTIONS
====================

1. The model has some false alarms (normal transactions flagged as fraud).
   In a real banking system, what is the cost of a false alarm?
   Is it better to have more false alarms or more missed frauds?
   How would you adjust the 'contamination' parameter to control this?

2. We used 3 features: amount, hour, merchant_freq.
   What other transaction features would improve fraud detection?
   Think about what makes a transaction suspicious in real life.

3. Isolation Forest is unsupervised — it doesn't use the 'is_fraud' label.
   If you had labelled fraud data (like the 'is_fraud' column here),
   would you still use Isolation Forest or switch to a supervised model?
   What's the tradeoff?

4. In the sales dataset exercise (Steps 2-8), we REMOVED outliers.
   In this fraud exercise, we KEPT and FLAGGED them.
   What was the deciding factor that changed the strategy?
"""
print(questions)

---

## Interview Corner

**Q: You detect that 8% of your training data are outliers. What do you do?**

**What they're testing:** Whether you blindly remove outliers or think in context.

**Answer direction:**
Start with domain understanding, not statistics. Are these errors or real extreme events? If errors, correct or remove them. If real extreme events, ask whether the model needs to perform well on them. In fraud detection, keeping them is mandatory. In house price prediction for typical buyers, removing or capping is fine. For production pipelines, never just drop rows — build a flagging step that routes outliers to a separate handler. Mention that 8% is high enough that removing them will noticeably shrink the dataset, so Winsorizing or a robust model (HuberRegressor, quantile regression) might be better than removal. If you mention checking for class imbalance implications when removing outliers from a classification dataset, you're in the top 10% of answers they hear.

---

## ML Spotlight

**PyOD (Python Outlier Detection)** is the most comprehensive open source library for outlier detection in ML pipelines. It wraps 40+ algorithms including Isolation Forest, LOF (Local Outlier Factor), COPOD, and ECOD in a single sklearn-compatible API. You switch between algorithms by changing one line.

Used in production at multiple fintech companies. 8k+ GitHub stars. Actively maintained.

```python
# Quick example — same interface for any algorithm
from pyod.models.iforest import IForest
from pyod.models.lof import LOF

# Switch algorithms without changing anything else
clf = IForest(contamination=0.04)
# clf = LOF(contamination=0.04)

clf.fit(X_train)
labels = clf.predict(X_test)  # 0 = inlier, 1 = outlier
```

GitHub: https://github.com/yzhao062/pyod

---

## Summary

| Method | Distribution Assumption | Multivariate | Best For |
|--------|------------------------|--------------|----------|
| IQR | None | No | Quick univariate check, skewed data |
| Z-Score | Normal | No | Normally distributed features |
| Isolation Forest | None | Yes | Complex datasets, fraud, anomaly detection |

**The rule that matters most:** The statistical method tells you *what* is an outlier. Your domain knowledge tells you *what to do about it*.

---

**Tomorrow: Day 7 — Data Cleaning**

Duplicates, inconsistent formatting, wrong data types, hidden whitespace. One dirty column silently breaks a model downstream. We'll fix all of it.

---
*42 Days of ML Challenge | https://github.com/VaishnaviJagtap18/42-Days-0f-ML-Challenge*